# Support Vector Regression 

### 1) Training 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

X = pd.read_csv("../X_train_log_knn.csv", index_col="id")
y = pd.read_csv("../y_train.csv", index_col="id").loc[X.index, "y"]   

cv = KFold(n_splits=5, shuffle=True, random_state=42)  
print(f"train {X.shape} | y {y.shape}")

train (1152, 216) | y (1152,)


In [2]:
# tune C, gamma, epsilon with a random search (scaler fitted inside each fold, y standardized during training)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

svr = TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), SVR(kernel="rbf")),
                                 transformer=StandardScaler())
param_space = {
    "regressor__svr__C":       loguniform(0.1, 100),     # penalty on errors: higher fits closer
    "regressor__svr__gamma":   loguniform(5e-4, 5e-2),   # kernel width: higher = more local (1/216 ≈ 0.005)
    "regressor__svr__epsilon": loguniform(0.01, 0.5),    # errors below epsilon are ignored (standardized units)
}
search = RandomizedSearchCV(svr, param_space, n_iter=60, scoring="r2", cv=cv, n_jobs=-1, random_state=0)
search.fit(X, y)

results = (pd.DataFrame(search.cv_results_)
           .rename(columns=lambda c: c.replace("param_regressor__svr__", ""))
           .sort_values("mean_test_score", ascending=False))
best = results.iloc[0]
print(f"best CV R2: {best['mean_test_score']:.4f} ± {best['std_test_score'] / np.sqrt(cv.get_n_splits()):.4f} (SE)")
display(results[["C", "gamma", "epsilon", "mean_test_score", "std_test_score"]].head(10).round(4))


best CV R2: 0.7057 ± 0.0092 (SE)


,C,gamma,epsilon,mean_test_score,std_test_score
56,15.4872,0.0055,0.0196,0.7057,0.0206
28,5.0096,0.0056,0.0282,0.7046,0.0205
13,11.1035,0.0037,0.0408,0.7025,0.0264
9,68.2349,0.0034,0.0770,0.6990,0.0277
52,26.4215,0.0033,0.0927,0.6977,0.0282
35,5.3515,0.0070,0.1014,0.6908,0.0190
55,36.9326,0.0026,0.0105,0.6904,0.0330
58,24.0488,0.0025,0.0240,0.6877,0.0336
41,8.5501,0.0082,0.0525,0.6856,0.0176
11,5.0734,0.0086,0.0108,0.6853,0.0173


In [3]:
# refined search around the first optimum 

param_space_fine = {
    "regressor__svr__C":       loguniform(2, 50),
    "regressor__svr__gamma":   loguniform(2e-3, 1e-2),
    "regressor__svr__epsilon": loguniform(1e-3, 5e-2),
}
search_fine = RandomizedSearchCV(svr, param_space_fine, n_iter=60, scoring="r2", cv=cv, n_jobs=-1, random_state=1)
search_fine.fit(X, y)

results_fine = (pd.DataFrame(search_fine.cv_results_)
                .rename(columns=lambda c: c.replace("param_regressor__svr__", ""))
                .sort_values("mean_test_score", ascending=False))
best_fine = results_fine.iloc[0]
print(f"first search : {search.best_score_:.4f}")
print(f"refined      : {best_fine['mean_test_score']:.4f} ± {best_fine['std_test_score'] / np.sqrt(cv.get_n_splits()):.4f} (SE)")
display(results_fine[["C", "gamma", "epsilon", "mean_test_score", "std_test_score"]].head(10).round(4))


first search : 0.7057
refined      : 0.7074 ± 0.0097 (SE)


,C,gamma,epsilon,mean_test_score,std_test_score
18,9.7324,0.0050,0.0012,0.7074,0.0217
21,7.5863,0.0047,0.0012,0.7072,0.0227
5,17.3103,0.0049,0.0051,0.7072,0.0221
33,14.5801,0.0047,0.0036,0.7071,0.0229
53,30.5384,0.0044,0.0026,0.7066,0.0238
46,32.2262,0.0049,0.0186,0.7065,0.0222
29,44.6497,0.0054,0.0134,0.7062,0.0207
3,11.3309,0.0060,0.0052,0.7051,0.0194
7,45.1442,0.0061,0.0034,0.7049,0.0193
58,11.7821,0.0059,0.0185,0.7047,0.0198


In [4]:
# Stacking: out-of-fold predictions of the tuned SVR on the shared folds (5 models, each trained on 4/5 of the rows)

from sklearn.base import clone
from sklearn.metrics import r2_score

model = clone(max([search, search_fine], key=lambda s: s.best_score_).best_estimator_)   # tuned SVR

fold = pd.read_csv("../Stacking/folds.csv", index_col="id").loc[X.index, "fold"]
oof = pd.Series(np.nan, index=X.index, name="svr")
for f in range(5):
    train, val = fold != f, fold == f
    model.fit(X[train], y[train])
    oof[val] = model.predict(X[val])

print(f"out-of-fold R2: {r2_score(y, oof):.4f}")
oof.to_csv("../Stacking/oof_svr.csv")


out-of-fold R2: 0.7083


In [5]:
# Full model: the tuned SVR trained on all 1152 training rows, predicting the 776 test rows (saved in the SVR folder, read by Stacking/stacking.ipynb)

X_test = pd.read_csv("../X_test_log_knn.csv", index_col="id")

model.fit(X, y)
pd.Series(model.predict(X_test), index=X_test.index, name="y").to_csv("submission_svr.csv")

### Log transform test (results recorded)

Same SVR (tuned on `X_train_knn`), same shared folds, `knn` vs `log_knn` (log of the 11 right-skewed features, then the kNN fill in log units):

| fold | knn | log_knn | diff |
|---|---|---|---|
| 0 | 0.7329 | 0.7426 | +0.0097 |
| 1 | 0.6846 | 0.7123 | +0.0277 |
| 2 | 0.6714 | 0.6751 | +0.0036 |
| 3 | 0.6868 | 0.7003 | +0.0135 |
| 4 | 0.6873 | 0.7067 | +0.0193 |

**Mean diff: +0.0148 ± 0.0041 (SE), log better in 5/5 folds** → the SVR now uses `log_knn`.